In [ ]:
#@title Installing vllm for fast Inference

!pip install vllm

In [ ]:
#@title Imports

import torch
import sys
import json

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"Python: {sys.version}")

PyTorch: 2.9.0+cu126
CUDA: 12.6
Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]


In [ ]:
#@title Imports & File Paths

from vllm import LLM, SamplingParams
import json
import re
import os
import gc

In [ ]:
#@title LLM Initialization & Sampling Parameters

llm = LLM(
    model="google/medgemma-4b-it",
    trust_remote_code=True,
    max_model_len=8192,
    gpu_memory_utilization=0.92,
    tensor_parallel_size=1,
    dtype="bfloat16"
)

sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.95,
    max_tokens=1024,
    stop_token_ids=[llm.get_tokenizer().eos_token_id]
)

print("MedGemma-4B loaded and ready!")

INFO 11-27 09:26:07 [utils.py:253] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'max_model_len': 8192, 'gpu_memory_utilization': 0.92, 'disable_log_stats': True, 'model': 'google/medgemma-4b-it'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

INFO 11-27 09:26:26 [model.py:631] Resolved architecture: Gemma3ForConditionalGeneration
INFO 11-27 09:26:26 [model.py:1745] Using max model len 8192
INFO 11-27 09:26:29 [scheduler.py:216] Chunked prefill is enabled with max_num_batched_tokens=8192.


...

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

WARNING 11-27 09:26:42 [system_utils.py:103] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 11-27 09:29:19 [llm.py:352] Supported tasks: ['generate']
MedGemma-4B loaded and ready!


In [ ]:
#@title Loading the JSON file

entries = []
input_file = "/content/qwen_generated_prompts_trained.json"

try:
    with open(input_file, "r", encoding="utf-8") as f:
        entries = json.load(f)
except FileNotFoundError:
    print(f"Error: File {input_file} not found. Please check the path.")
    entries = []
except json.JSONDecodeError as e:
    print(f"Error: Failed to parse JSON in {input_file}: {e}")
    entries = []

# Filter out invalid entries
valid_entries = []
for entry in entries:
    if isinstance(entry, dict) and all(key in entry for key in ["qid", "original_prompt", "generated_prompt"]):
        valid_entries.append(entry)
    else:
        print(f"Warning: Skipping invalid entry (missing required keys or not a dict): {entry.get('qid', 'unknown')}")

print(f"Successfully loaded {len(valid_entries):,} valid entries")
if valid_entries:
    print("Sample keys:", list(valid_entries[0].keys()))
    print("Sample qid:", valid_entries[0].get("qid"))

entries = valid_entries

Successfully loaded 743 valid entries
Sample keys: ['qid', 'original_prompt', 'generated_prompt']
Sample qid: 2880


In [ ]:
#@title Sample Entry
entries[0]

{'qid': 2880,
 'original_prompt': 'How has the evolution of bioluminescence in marine organisms contributed to their survival and success in their respective ecosystems?',
 'generated_prompt': 'Discuss the evolutionary significance of bioluminescence in marine organisms such as jellyfish, octopuses, and squid. Provide examples of how bioluminescence has influenced the behavior and survival of these species over time. Use real-world evidence to support your arguments and consider the ecological implications of bioluminescent behaviors on the overall health and diversity of marine ecosystems.'}

In [ ]:
#@title Build full MedGemma chat format for every entry
template = "<start_of_turn>user\n{}\n<end_of_turn>\n<start_of_turn>model"

all_prompts = []
prompt_to_entry_index = []  # Maps prompts to entry indices
prompt_type_list = []       # Tracks 'original' or 'generated'

for idx, entry in enumerate(entries):
    qid = entry.get("qid")
    orig_prompt = entry.get("original_prompt", "").strip()
    gen_prompt = entry.get("generated_prompt", "").strip()

    # Skip if both prompts are empty
    if not orig_prompt and not gen_prompt:
        print(f"Skipping qid {qid} (idx {idx}): both prompts empty")
        continue

    # Add original prompt if not empty
    if orig_prompt:
        all_prompts.append(template.format(orig_prompt))
        prompt_to_entry_index.append(idx)
        prompt_type_list.append("original")
    else:
        print(f"Warning: qid {qid} has empty original_prompt")

    # Add generated prompt if not empty
    if gen_prompt:
        all_prompts.append(template.format(gen_prompt))
        prompt_to_entry_index.append(idx)
        prompt_type_list.append("generated")
    else:
        print(f"Warning: qid {qid} has empty generated_prompt")

print(f"Prepared {len(all_prompts):,} prompts ({sum(t=='original' for t in prompt_type_list)} original + {sum(t=='generated' for t in prompt_type_list)} generated)")
if all_prompts:
    print("Sample prompt:", all_prompts[0][:100] + "...")

Prepared 1,486 prompts (743 original + 743 generated)
Sample prompt: <start_of_turn>user
How has the evolution of bioluminescence in marine organisms contributed to thei...


In [ ]:
#@title Generating Responses

print("Starting inference with MedGemma-4B-IT...")
outputs = llm.generate(all_prompts, sampling_params, use_tqdm=True)
print(f"Generated {len(outputs)} responses")

Starting inference with MedGemma-4B-IT...


Adding requests:   0%|          | 0/1486 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1486 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

Generated 1486 responses


In [ ]:
#@title Initialize results
results = [None] * len(entries)

# Process outputs and group by entry
for prompt_idx, output in enumerate(outputs):
    entry_idx = prompt_to_entry_index[prompt_idx]
    prompt_type = prompt_type_list[prompt_idx]
    response_text = output.outputs[0].text.strip()

    # Clean up response
    if response_text.endswith("<end_of_turn>"):
        response_text = response_text[:-len("<end_of_turn>")].strip()

    # Create base structure if not exists
    if results[entry_idx] is None:
        results[entry_idx] = {
            "qid": entries[entry_idx].get("qid"),
            "original": {"prompt": "", "response": ""},
            "generated": {"prompt": "", "response": ""}
        }

    # Fill in the correct field
    if prompt_type == "original":
        results[entry_idx]["original"]["prompt"] = entries[entry_idx].get("original_prompt", "").strip()
        results[entry_idx]["original"]["response"] = response_text
    else:  # generated
        results[entry_idx]["generated"]["prompt"] = entries[entry_idx].get("generated_prompt", "").strip()
        results[entry_idx]["generated"]["response"] = response_text

# Filter out None entries
final_results = [r for r in results if r is not None]

In [ ]:
#@title Save to JSONL
output_file = "/content/medgemma_inference_responses.jsonl"
with open(output_file, "w", encoding="utf-8") as f:
    for item in final_results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved {len(final_results)} processed entries to {output_file}")

Saved 743 processed entries to /content/medgemma_inference_responses.jsonl


In [ ]:
#@title Sample Output

print("Sample output:")
print(json.dumps(final_results[0], indent=2, ensure_ascii=False))

Sample output:
{
  "qid": 2880,
  "original": {
    "prompt": "How has the evolution of bioluminescence in marine organisms contributed to their survival and success in their respective ecosystems?",
    "response": "Bioluminescence, the production and emission of light by living organisms, has played a crucial role in the survival and success of many marine organisms. Here's how:\n\n**1. Predator Avoidance:**\n\n*   **Startle Response:** A sudden flash of light can startle or disorient a predator, allowing the prey to escape. This is particularly effective against visually-oriented predators.\n*   **Counterillumination:** Many deep-sea organisms, like hatchetfish and lanternfish, have photophores on their ventral sides that emit a faint, downward-facing light. This light matches the downwelling sunlight, effectively camouflaging them from predators looking up from below. The predator sees a uniform background, making it difficult to distinguish the organism from the darkness.\n*   **B